# SQD Post processing

In [ ]:
import os
import json
import time
import pickle
import logging

import numpy as np
import matplotlib.pyplot as plt

from functools import partial
from pathlib import Path

import pyscf
from pyscf import gto, scf, mp, mcscf, cc, ci, tools, ao2mo
from pyscf.scf import chkfile
from pyscf import lib
from pyscf.tools import fcidump

from qiskit_ibm_runtime import RuntimeDecoder

from qiskit_addon_sqd.fermion import (
    SCIResult,
    diagonalize_fermionic_hamiltonian,
    solve_sci_batch,
)

# ──────────────────────────────────────────────────────────────────────────────
# Logging
# ──────────────────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

## Initial parameters

In [ ]:
shift_array=['-0.15','0.0','0.15','0.3','0.45','0.6','0.75','0.9','1.05','1.2']
shift='0.0'#

charge=3
spin_number=4
medium='gas'

num_orbitals=17
num_elec_a,num_elec_b=13,9
num_qubits=num_orbitals*2

backend_name='ibm_pittsburgh'

CLASSICAL_DATA_DIR=f'../../classical_methods/{medium}/charge{charge}_spin{spin_number}'
RESULTS_DIR=f'../../sampler_runs/{medium}/charge{charge}_spin{spin_number}/{num_qubits}qubits'
as_label = 'norb_%d_nelec_%d_%d'%(num_orbitals,num_elec_a,num_elec_b)

SQD_RESULTS_DIR= f'charge{charge}_spin{spin_number}/{num_qubits}qubits/shift_{shift}'
Path(SQD_RESULTS_DIR).mkdir(parents=True,exist_ok=True)

## Load results of classical calculations

In [ ]:
def load_converged_scf(CLASSICAL_DATA_DIR,shift,medium):
    """Load a converged ROHF calculation from a checkpoint file."""

    shift_sign = 'minus_' if shift[0]=='-' else ''
    shift_fname_core = shift_sign + shift.strip('-').split('.')[0] + '_' + shift.split('.')[1][::-1].zfill(2)[::-1]
    chkfile_path=f'{CLASSICAL_DATA_DIR}/chk_files/shift_{shift_fname_core}.chk'
    mol = lib.chkfile.load_mol(Path(chkfile_path))
    mf = pyscf.scf.ROHF(mol).density_fit()
    mo_avas=None
    if medium=='gas':
        mo_coeff=np.load(f'{CLASSICAL_DATA_DIR}/orbitals/shift_{shift}/mo_coeff_{as_label}.npy')
        mf.__dict__.update(chkfile.load(chkfile_path, 'scf'))
        mf.converged=True
        mf.mo_coeff=mo_coeff
    if medium=='solvent':
        mo_coeff=np.load(f'{CLASSICAL_DATA_DIR}/orbitals/shift_{shift}/mo_coeff_{as_label}.npy')
        mf = mf.PCM()
        mf.with_solvent.method = 'IEF-PCM'
        mf.with_solvent.eps = 78.3553
        mf.__dict__.update(chkfile.load(chkfile_path, 'scf'))
        mf.converged=True
        mf.mo_coeff=mo_coeff
        mc = mcscf.CASCI(mf, ncas=num_orbitals, nelecas=(num_elec_a, num_elec_b))
        mc = mc.PCM()
        mc.with_solvent.method = mf.with_solvent.method
        mc.with_solvent.eps = mf.with_solvent.eps
        mc.mo_coeff=mo_coeff
        mo_avas=mo_coeff
        return mf,mc,mo_avas
    return mf

def load_converged_scf_and_integrals(CLASSICAL_DATA_DIR,shift,medium):
    """Load a converged ROHF calculation from a checkpoint file. Does not require fcidump."""

    shift_sign = 'minus_' if shift[0]=='-' else ''
    shift_fname_core = shift_sign + shift.strip('-').split('.')[0] + '_' + shift.split('.')[1][::-1].zfill(2)[::-1]
    chkfile_path=f'{CLASSICAL_DATA_DIR}/chk_files/shift_{shift_fname_core}.chk'
    mol = lib.chkfile.load_mol(Path(chkfile_path))
    mf = pyscf.scf.ROHF(mol).density_fit()
    mo_avas=[]
    h1e,h2e,ecore=[],[],[]
    mycas=[]
    if medium=='gas':
        mo_coeff=np.load(f'{CLASSICAL_DATA_DIR}/orbitals/shift_{shift}/mo_coeff_{as_label}.npy')
        mf.__dict__.update(chkfile.load(chkfile_path, 'scf'))
        mf.converged=True
        mf.mo_coeff=mo_coeff

        mycas = mcscf.CASCI(mf, ncas=num_orbitals, nelecas=(num_elec_a, num_elec_b))
        mycas.mo_coeff = np.load(f'{CLASSICAL_DATA_DIR}/orbitals/shift_{shift}/mo_coeff_{as_label}.npy')
        h1e, ecore = mycas.get_h1eff()
        h2e_comp = mycas.get_h2eff()
        h2e = ao2mo.restore(1, h2e_comp, mycas.ncas)

    if medium=='solvent':
        mo_coeff=np.load(f'{CLASSICAL_DATA_DIR}/orbitals/shift_{shift}/mo_coeff_{as_label}.npy')
        mf = mf.PCM()
        mf.with_solvent.method = 'IEF-PCM'
        mf.with_solvent.eps = 78.3553
        mf.__dict__.update(chkfile.load(chkfile_path, 'scf'))
        mf.converged=True
        mf.mo_coeff=mo_coeff

        mycas = mcscf.CASCI(mf, ncas=num_orbitals, nelecas=(num_elec_a, num_elec_b)).PCM()
        mycas.with_solvent.method = 'IEF-PCM'
        mycas.with_solvent.eps = 78.3553
        mycas.with_solvent.frozen = True   # <- important, see below
        mycas.mo_coeff = np.load(f'{CLASSICAL_DATA_DIR}/orbitals/shift_{shift}/mo_coeff_{as_label}.npy')
        h1e, ecore = mycas.get_h1eff()       # includes PCM reaction-field operator
        h2e_comp   = mycas.get_h2eff()       # PCM does NOT touch the 2e part
        h2e        = ao2mo.restore(1, h2e_comp, mycas.ncas)
    print(type(mf),type(mycas),type(h1e),type(h2e),type(ecore))
    return mf,mycas,h1e,h2e,ecore


def load_integrals(CLASSICAL_DATA_DIR,shift):
    fname = f'{CLASSICAL_DATA_DIR}/active_spaces/shift_%s/spin_%d_shift_%s_%s.fcidump'%(shift,spin_number,shift,as_label)
    data=fcidump.read(fname)
    h1e=data['H1']
    h2e_comp=data['H2']
    h2e=ao2mo.restore(1,h2e_comp,num_orbitals)
    ecore=data['ECORE']
    return h1e,h2e,ecore

def load_reference_energy(CLASSICAL_DATA_DIR,as_label,idx):
    results = np.load(f'{CLASSICAL_DATA_DIR}/active_space_results_classical_{as_label}.npy',allow_pickle=True).item()
    method='CCSD'
    ccsd_etot=results['%s_%s'%(as_label,method)]
    #print(ccsd_etot)
    return ccsd_etot[idx]

In [ ]:
# mf=load_converged_scf(CLASSICAL_DATA_DIR,shift,medium) ### uses fcidump
# h1e_cas,h2e_cas,ecore=load_integrals(CLASSICAL_DATA_DIR,shift) ### uses fcidump.
mf,mc,h1e_cas,h2e_cas,ecore=load_converged_scf_and_integrals(CLASSICAL_DATA_DIR,shift,medium) ### Without fcidump.
nuclear_repulsion_energy=ecore#mf.mol.energy_nuc()
reference_energy= load_reference_energy(CLASSICAL_DATA_DIR,as_label,shift_array.index(shift))#CCSD energy

## Load Hardware Results

In [ ]:
def load_hardware_results(
    results_dir: str,
    num_qubits: int,
    shift: float,
    backend_name: str,
):
    results_file=str(RESULTS_DIR)+f'/qubits_{num_qubits}_shift_{shift}_{backend_name}.json'

    # results_file = os.path.join(
    #     results_dir,
    #     f"qubits_{num_qubits}_shift_{shift}_{backend_name}.json",
    # )
    log.info("Loading hardware results from %s", results_file)

    with open(results_file) as f:
        pub_result = json.load(f, cls=RuntimeDecoder)

    num_shots = pub_result.data.meas.num_shots
    num_bits = pub_result.data.meas.num_bits
    log.info(
        "Loaded BitArray: %d shots × %d bits", num_shots, num_bits
    )

    return pub_result


## SQD

### SQD parameters

In [ ]:
max_iterations: int = 5
energy_tol: float = 1e-5
occupancies_tol: float = 1e-3
num_batches: int = 5
samples_per_batch: int = 200
symmetrize_spin: bool = not bool(spin_number)
carryover_threshold: float = 1e-4
max_cycle: int = 200
spin_sq: float = mf.spin_square()[0]
seed: int = 12345

### SQD Post processing loop

In [ ]:
def run_sqd(
    h1e: np.ndarray,
    h2e: np.ndarray,
    bit_array,
    num_orbitals: int,
    num_elec_a: int,
    num_elec_b: int,
    nuclear_repulsion_energy: float,
    # S-CORE options
    max_iterations: int = 10,
    energy_tol: float = 1e-3,
    occupancies_tol: float = 1e-3,
    # Batch / solver options
    num_batches: int = 3,
    samples_per_batch: int = 300,
    symmetrize_spin: bool = True,
    carryover_threshold: float = 1e-4,
    max_cycle: int = 200,
    spin_sq: float = 0.0,
    seed: int = 12345,
):

    # Configure the SCI solver with the target ⟨S²⟩ and max Davidson cycles
    sci_solver = partial(
        solve_sci_batch,
        spin_sq=spin_sq,
        max_cycle=max_cycle,
    )

    # Accumulator for intermediate results from each S-CORE iteration
    result_history: list[list[SCIResult]] = []
    t_start = time.time()

    def callback(results: list[SCIResult]):
        """
        Called at the end of each S-CORE iteration with the batch results.
        Logs energies and subspace dimensions for monitoring convergence.
        """
        result_history.append(results)
        iteration = len(result_history)
        log.info("─── S-CORE iteration %d ───", iteration)
        for i, res in enumerate(results):
            e_total = res.energy + nuclear_repulsion_energy
            dim = np.prod(res.sci_state.amplitudes.shape)
            elapsed = (time.time() - t_start) / 60.0
            log.info(
                "  Batch %d  │  E_total = %.10f Ha  │  dim = %d  │  %.1f min",
                i, e_total, dim, elapsed,
            )

    # ── Main SQD call ──────────────────────────────────────────────────────
    log.info("Starting SQD post-processing (%d orbitals, %d↑ + %d↓ electrons)",
             num_orbitals, num_elec_a, num_elec_b)
    log.info("  max_iterations=%d, num_batches=%d, samples_per_batch=%d",
             max_iterations, num_batches, samples_per_batch)

    result = diagonalize_fermionic_hamiltonian(
        h1e,
        h2e,
        bit_array,
        samples_per_batch=samples_per_batch,
        norb=num_orbitals,
        nelec=(num_elec_a, num_elec_b),
        num_batches=num_batches,
        energy_tol=energy_tol,
        occupancies_tol=occupancies_tol,
        max_iterations=max_iterations,
        sci_solver=sci_solver,
        symmetrize_spin=symmetrize_spin,
        carryover_threshold=carryover_threshold,
        callback=callback,
        seed=seed,
    )

    wall_time = time.time() - t_start
    best_energy = result.energy + nuclear_repulsion_energy
    log.info("SQD complete in %.1f min  │  Best E_total = %.10f Ha",
             wall_time / 60.0, best_energy)

    return result, result_history, wall_time

## Analysis Functions

In [ ]:
def analyze_convergence(
    result_history: list[list[SCIResult]],
    nuclear_repulsion_energy: float,
    reference_energy: float = None,
):
    """
    Extract per-iteration best energies and compute energy errors
    relative to a classical reference (if provided).

    Returns
    -------
    iterations : list[int]
        0-indexed iteration numbers.
    best_energies : list[float]
        Best (min over batches) total energy at each iteration.
    energy_errors : list[float] or None
        |E_SQD - E_ref| at each iteration.  None if no reference given.
    """
    iterations = list(range(len(result_history)))
    best_energies = [
        min(batch, key=lambda r: r.energy).energy + nuclear_repulsion_energy
        for batch in result_history
    ]

    energy_errors = None
    if reference_energy is not None:
        energy_errors = [abs(e - reference_energy) for e in best_energies]

    return iterations, best_energies, energy_errors


def plot_results(
    result: SCIResult,
    result_history: list[list[SCIResult]],
    nuclear_repulsion_energy: float,
    reference_energy: float = None,
    save_path: str = None,
):
    """
    Produce the standard two-panel SQD diagnostic figure:
        Left  — energy error vs. S-CORE iteration (log scale)
        Right — average spatial orbital occupancy after final iteration

    Parameters
    ----------
    result : SCIResult
        Final (best) SQD result carrying orbital_occupancies.
    result_history : list[list[SCIResult]]
        Full iteration history.
    nuclear_repulsion_energy : float
        Constant offset for total energies.
    reference_energy : float, optional
        Classical reference energy for error computation.
    save_path : str, optional
        If provided, save figure to this path instead of displaying.
    """
    iters, best_e, e_errors = analyze_convergence(
        result_history, nuclear_repulsion_energy, reference_energy
    )

    fig, axs = plt.subplots(1, 2, figsize=(14, 6))

    # ── Left panel: energy convergence ─────────────────────────────────────
    if e_errors is not None:
        axs[0].plot(iters, e_errors, marker="o", linewidth=2, label="Energy error")
        axs[0].set_yscale("log")
        axs[0].set_ylim(bottom=1e-5)

        # Chemical accuracy line (1 kcal/mol ≈ 1.6 mHa)
        chem_accuracy_ha = 1.6e-3
        axs[0].axhline(
            y=chem_accuracy_ha,
            color="#BF5700",
            linestyle="--",
            linewidth=1.5,
            label="Chemical accuracy (1.6 mHa)",
        )
        axs[0].set_ylabel("| E_SQD − E_ref |  (Ha)", fontsize=12)
    else:
        # No reference — just plot raw energies
        axs[0].plot(iters, best_e, marker="o", linewidth=2, label="Best E_total")
        axs[0].set_ylabel("Total Energy (Ha)", fontsize=12)

    axs[0].set_xticks(iters)
    axs[0].set_xlabel("S-CORE Iteration", fontsize=12)
    axs[0].set_title("Ground-State Energy Convergence")
    axs[0].legend(fontsize=10)
    axs[0].grid(True, alpha=0.3)

    # ── Right panel: orbital occupancy ─────────────────────────────────────
    # result.orbital_occupancies has shape (2, num_orbitals) for (alpha, beta)
    occ = np.sum(result.orbital_occupancies, axis=0)  # sum over spin
    orb_idx = np.arange(len(occ))

    axs[1].bar(orb_idx, occ, width=0.7, color="#4C72B0", edgecolor="white")
    axs[1].set_xticks(orb_idx)
    axs[1].set_xlabel("Spatial Orbital Index", fontsize=12)
    axs[1].set_ylabel("Avg. Occupancy (α + β)", fontsize=12)
    axs[1].set_title("Orbital Occupancies (Final Iteration)")
    axs[1].set_ylim(0, 2.1)
    axs[1].grid(True, axis="y", alpha=0.3)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
        log.info("Figure saved to %s", save_path)
    else:
        plt.show()


def print_summary(
    result: SCIResult,
    result_history: list[list[SCIResult]],
    nuclear_repulsion_energy: float,
    reference_energy: float = None,
    wall_time: float = None,
):
    """Print a concise summary table to stdout."""
    iters, best_e, e_errors = analyze_convergence(
        result_history, nuclear_repulsion_energy, reference_energy
    )

    print("\n" + "=" * 72)
    print("  SQD POST-PROCESSING SUMMARY")
    print("=" * 72)
    print(f"  Active space       : ({num_elec_a + num_elec_b}e, {num_orbitals}o)")
    print(f"  S-CORE iterations  : {len(result_history)}")
    print(f"  Batches / iteration: {num_batches}")
    print(f"  Samples per batch  : {samples_per_batch}")

    if wall_time is not None:
        print(f"  Wall-clock time    : {wall_time / 60:.1f} min")

    print("-" * 72)
    print(f"  {'Iter':>4s}  {'Best E_total (Ha)':>22s}", end="")
    if e_errors is not None:
        print(f"  {'Error (Ha)':>14s}  {'Error (kcal/mol)':>16s}", end="")
    print()
    print("-" * 72)

    for i, e in enumerate(best_e):
        line = f"  {i:4d}  {e:22.10f}"
        if e_errors is not None:
            err_ha = e_errors[i]
            err_kcal = err_ha * 627.509  # 1 Ha = 627.509 kcal/mol
            line += f"  {err_ha:14.6e}  {err_kcal:16.4f}"
        print(line)

    print("-" * 72)
    final_e = best_e[-1]
    print(f"  Final best energy  : {final_e:.10f} Ha")

    if reference_energy is not None:
        final_err = e_errors[-1]
        print(f"  Reference energy   : {reference_energy:.10f} Ha")
        print(f"  Final error        : {final_err:.6e} Ha"
              f"  ({final_err * 627.509:.4f} kcal/mol)")
        if final_err < 1.6e-3:
            print("  ✓ Within chemical accuracy (1 kcal/mol ≈ 1.6 mHa)")
        else:
            print("  ✗ Not yet at chemical accuracy — consider increasing"
                  " samples_per_batch or max_iterations")

    print("=" * 72 + "\n")


## MAIN

In [ ]:
# if __name__ == "__main__":

# ── Validate inputs ────────────────────────────────────────────────────
if h1e_cas is None or h2e_cas is None:
    raise ValueError(
        "Active-space integrals h1e_cas and h2e_cas must be set.\n"
        "Load them from your upstream checkpoint in the 'Load results of classical calculations' section above."
    )

if nuclear_repulsion_energy == 0.0:
    log.warning(
        "nuclear_repulsion_energy is 0.0 — did you forget to set it? "
        "The reported total energies will be missing the nuclear + core offset."
    )

# ── Load quantum samples ───────────────────────────────────────────────
pub_result = load_hardware_results(
    RESULTS_DIR, num_qubits, shift, backend_name
)
bit_array = pub_result.data.meas

# ── Run the S-CORE + diagonalization ───────────────────────────────────
result, result_history, wall_time = run_sqd(
    h1e=h1e_cas,
    h2e=h2e_cas,
    bit_array=bit_array,
    num_orbitals=num_orbitals,
    num_elec_a=num_elec_a,
    num_elec_b=num_elec_b,
    nuclear_repulsion_energy=nuclear_repulsion_energy,
    max_iterations=max_iterations,
    energy_tol=energy_tol,
    occupancies_tol=occupancies_tol,
    num_batches=num_batches,
    samples_per_batch=samples_per_batch,
    symmetrize_spin=symmetrize_spin,
    carryover_threshold=carryover_threshold,
    max_cycle=max_cycle,
    spin_sq=spin_sq,
    seed=seed,
)

# ── Save full result history ───────────────────────────────────────────
out_pkl = os.path.join(
    SQD_RESULTS_DIR,
    f"sqd_history_qubits_{num_qubits}_shift_{shift}_{backend_name}.pkl",
)
with open(out_pkl, "wb") as f:
    pickle.dump(result_history, f)
log.info("Result history saved to %s", out_pkl)

# ── Print summary + plot ───────────────────────────────────────────────
print_summary(
    result, result_history,
    nuclear_repulsion_energy, reference_energy, wall_time,
)

plot_path = os.path.join(
    SQD_RESULTS_DIR,
    f"sqd_convergence_qubits_{num_qubits}_shift_{shift}_{backend_name}.png",
)
plot_results(
    result, result_history,
    nuclear_repulsion_energy, reference_energy,
    save_path=plot_path,
)
